# Smart Resume Analyzer with AI-Based Feedback
### AI Capstone Project — SkillOrbit

**Modules implemented:**
1. Resume Upload & Parsing (PDF / DOCX)
2. Resume Score Analyzer (rule-based AI scoring out of 100)
3. ATS Keyword Checker (role-based keyword matching using NLP)
4. Smart Feedback System (improvement suggestions)
5. Dashboard & Report Generation (visual summary)

This notebook is designed to run top-to-bottom in **Google Colab**.
Run each cell in order. When prompted, upload your resume (PDF or DOCX).

**Kept intentionally simple:** everything is plain Python — file reading libraries
(`pdfplumber`, `python-docx`), regular expressions (`re`) for keyword/pattern
matching, and `matplotlib` for the dashboard charts. No deep learning, no heavy
NLP libraries, no external AI APIs — matching the project's own scope limits.


## Step 0: Install & Import Required Libraries

In [ ]:
# Install required libraries
# pdfplumber   -> extract text from PDF resumes
# python-docx  -> extract text from DOCX resumes
# (matplotlib and re/string/io are already built into Colab / Python)
!pip install pdfplumber python-docx -q


In [ ]:
# Import libraries
import re
import io
import string
import pdfplumber
import docx
import matplotlib.pyplot as plt

from google.colab import files   # for file upload widget in Colab

print("All libraries loaded successfully.")


## Module 1: Resume Upload and Parsing
**Functionalities:** upload resume file, extract text from PDF/DOCX, store extracted content.

**Learning outcomes:** file handling, text extraction, backend integration.


In [ ]:
def extract_text_from_pdf(file_bytes):
    """
    Extract raw text from a PDF file using pdfplumber.
    file_bytes: bytes object of the uploaded PDF file
    Returns: extracted text as a single string
    """
    text = ""
    with pdfplumber.open(io.BytesIO(file_bytes)) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    return text


def extract_text_from_docx(file_bytes):
    """
    Extract raw text from a DOCX file using python-docx.
    file_bytes: bytes object of the uploaded DOCX file
    Returns: extracted text as a single string
    """
    document = docx.Document(io.BytesIO(file_bytes))
    text = "\n".join(paragraph.text for paragraph in document.paragraphs)
    return text


def upload_and_parse_resume():
    """
    Handles file upload in Colab and routes to the correct parser
    based on file extension (.pdf or .docx).
    Returns: (filename, extracted_text)
    """
    print("Please upload your resume (PDF or DOCX format)...")
    uploaded = files.upload()   # opens Colab's file picker

    if not uploaded:
        raise ValueError("No file was uploaded.")

    filename = list(uploaded.keys())[0]
    file_bytes = uploaded[filename]

    if filename.lower().endswith(".pdf"):
        text = extract_text_from_pdf(file_bytes)
    elif filename.lower().endswith(".docx"):
        text = extract_text_from_docx(file_bytes)
    else:
        raise ValueError("Unsupported file format. Please upload a PDF or DOCX file.")

    if not text.strip():
        raise ValueError("Could not extract any text. The file may be a scanned image PDF.")

    print(f"Successfully parsed '{filename}' ({len(text)} characters extracted).")
    return filename, text


# Run the upload step
resume_filename, resume_text = upload_and_parse_resume()

# Preview the first 500 characters of extracted text
print("\n--- Resume Text Preview ---\n")
print(resume_text[:500] + "...")


## Module 2: Resume Score Analyzer
**Analysis parameters:** resume structure, skills section, education details, projects section,
contact information, resume completeness.

**Output:** resume score out of 100 (rule-based AI logic).


In [ ]:
def check_contact_info(text):
    """
    Checks for presence of email and phone number using regex.
    Returns a score (0-15) and details of what was found/missing.
    """
    email_pattern = r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+"
    phone_pattern = r"(\+?\d{1,3}[-.\s]?)?\(?\d{3,5}\)?[-.\s]?\d{3}[-.\s]?\d{3,4}"

    has_email = bool(re.search(email_pattern, text))
    has_phone = bool(re.search(phone_pattern, text))

    score = 0
    score += 8 if has_email else 0
    score += 7 if has_phone else 0

    return score, {"email_found": has_email, "phone_found": has_phone}


def check_section_presence(text):
    """
    Checks for the presence of key resume sections using keyword matching.
    Each detected section contributes to the overall structure score.
    Sections checked: Education, Skills, Projects, Experience, Certifications.
    """
    text_lower = text.lower()

    sections = {
        "education":     ["education", "academic", "qualification"],
        "skills":        ["skills", "technical skills", "competencies"],
        "projects":      ["projects", "project work"],
        "experience":    ["experience", "internship", "work history"],
        "certifications":["certification", "certifications", "courses"],
    }

    found = {}
    for section, keywords in sections.items():
        found[section] = any(kw in text_lower for kw in keywords)

    # Weight: education, skills, projects are most important (15 each)
    # experience, certifications are bonus (10 each)
    weights = {"education": 15, "skills": 15, "projects": 15,
               "experience": 10, "certifications": 10}

    score = sum(weights[s] for s, present in found.items() if present)
    return score, found


def check_resume_completeness(text):
    """
    A basic heuristic for completeness/length: very short resumes
    are penalized as likely incomplete.
    Returns a score out of 15.
    """
    word_count = len(text.split())

    if word_count < 100:
        return 3, word_count      # too short -> likely incomplete
    elif word_count < 250:
        return 9, word_count      # borderline
    else:
        return 15, word_count     # sufficiently detailed


def analyze_resume_score(text):
    """
    Combines all sub-scores into a single Resume Score out of 100.
    Breakdown:
      - Contact Info:        15 pts
      - Section Presence:    65 pts (education/skills/projects/experience/certs)
      - Completeness:        20 pts  (recalculated below to sum to 100)
    """
    contact_score, contact_details = check_contact_info(text)
    section_score, section_details = check_section_presence(text)
    completeness_score, word_count = check_resume_completeness(text)

    # Normalize weights so total = 100
    # contact(15) + sections(65 max) + completeness(15 max) -> scale completeness to /20
    completeness_score_scaled = round((completeness_score / 15) * 20)

    total_score = contact_score + section_score + completeness_score_scaled
    total_score = min(total_score, 100)  # cap at 100

    breakdown = {
        "Contact Information (out of 15)": contact_score,
        "Resume Sections (out of 65)": section_score,
        "Completeness (out of 20)": completeness_score_scaled,
    }

    details = {
        "contact": contact_details,
        "sections": section_details,
        "word_count": word_count,
    }

    return total_score, breakdown, details


# Run resume scoring
resume_score, score_breakdown, score_details = analyze_resume_score(resume_text)

print(f"Overall Resume Score: {resume_score} / 100\n")
print("Score Breakdown:")
for k, v in score_breakdown.items():
    print(f"  - {k}: {v}")


## Module 3: ATS Keyword Checker
Users select a target job role, and the system compares resume keywords with
industry-required keywords for that role (simple text/keyword matching).


In [ ]:
# Role -> required keyword bank (expand as needed)
ROLE_KEYWORDS = {
    "Data Analyst": [
        "sql", "excel", "python", "power bi", "tableau", "data cleaning",
        "statistics", "pandas", "numpy", "data visualization", "reporting"
    ],
    "Web Developer": [
        "html", "css", "javascript", "react", "node.js", "rest api",
        "git", "responsive design", "mongodb", "sql"
    ],
    "AI Engineer": [
        "python", "machine learning", "deep learning", "tensorflow",
        "pytorch", "nlp", "computer vision", "scikit-learn", "data preprocessing"
    ],
    "Cloud Engineer": [
        "aws", "azure", "gcp", "docker", "kubernetes", "ci/cd",
        "terraform", "linux", "networking", "cloud security"
    ],
}


def preprocess_text(text):
    """
    Lowercases and strips punctuation for reliable keyword matching.
    """
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    return text


def check_ats_keywords(text, role):
    """
    Compares resume text against the required keyword list for the chosen role.
    Returns: matched keywords, missing keywords, and an ATS match percentage.
    """
    if role not in ROLE_KEYWORDS:
        raise ValueError(f"Role '{role}' not found. Choose from: {list(ROLE_KEYWORDS.keys())}")

    processed_text = preprocess_text(text)
    required_keywords = ROLE_KEYWORDS[role]

    matched = [kw for kw in required_keywords if kw in processed_text]
    missing = [kw for kw in required_keywords if kw not in processed_text]

    match_percentage = round((len(matched) / len(required_keywords)) * 100)

    return matched, missing, match_percentage


# ---- Select target role here ----
target_role = "AI Engineer"   # change to: "Data Analyst", "Web Developer", "Cloud Engineer"

matched_keywords, missing_keywords, ats_score = check_ats_keywords(resume_text, target_role)

print(f"Target Role: {target_role}")
print(f"ATS Compatibility Score: {ats_score}%\n")
print(f"Matched Keywords ({len(matched_keywords)}): {matched_keywords}")
print(f"Missing Keywords ({len(missing_keywords)}): {missing_keywords}")


## Module 4: Smart Feedback System
Generates rule-based improvement suggestions using the resume score breakdown
and ATS keyword gaps from Modules 2 and 3.


In [ ]:
def generate_feedback(score_breakdown, score_details, missing_keywords, ats_score):
    """
    Rule-based recommendation engine.
    Produces a list of clear, actionable suggestions for the user.
    """
    suggestions = []

    # --- Contact info feedback ---
    contact = score_details["contact"]
    if not contact["email_found"]:
        suggestions.append("Add a professional email address to your contact section.")
    if not contact["phone_found"]:
        suggestions.append("Add a phone number so recruiters can reach you.")

    # --- Section presence feedback ---
    sections = score_details["sections"]
    if not sections["education"]:
        suggestions.append("Add an Education section with your degree and institution.")
    if not sections["skills"]:
        suggestions.append("Add a dedicated Skills section listing your technical skills.")
    if not sections["projects"]:
        suggestions.append("Include a Projects section - this is heavily weighted by recruiters.")
    if not sections["experience"]:
        suggestions.append("Consider adding an Internship/Experience section, even short-term roles.")
    if not sections["certifications"]:
        suggestions.append("List relevant certifications or online courses to strengthen your profile.")

    # --- Completeness feedback ---
    if score_details["word_count"] < 250:
        suggestions.append("Your resume looks short - add more detail to projects and skills.")

    # --- ATS keyword feedback ---
    if missing_keywords:
        formatted = ", ".join(missing_keywords[:5])  # show top 5 to keep it actionable
        suggestions.append(f"Add these role-relevant keywords to improve ATS match: {formatted}.")

    if ats_score < 50:
        suggestions.append("Your resume has low keyword overlap with this role - consider tailoring it specifically for the job description.")

    if not suggestions:
        suggestions.append("Great job! Your resume covers all key areas well.")

    return suggestions


feedback_list = generate_feedback(score_breakdown, score_details, missing_keywords, ats_score)

print("Smart Feedback & Suggestions:\n")
for i, tip in enumerate(feedback_list, start=1):
    print(f"{i}. {tip}")


## Module 5: Dashboard and Report Generation
Displays all analysis results (resume score, ATS score, missing skills, suggestions)
through simple, interactive-style visualizations using matplotlib.


In [ ]:
def display_dashboard(filename, resume_score, score_breakdown, target_role,
                       ats_score, matched_keywords, missing_keywords, feedback_list):
    """
    Renders a visual dashboard summarizing the full resume analysis.
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # --- Chart 1: Resume score breakdown (bar chart) ---
    labels = list(score_breakdown.keys())
    values = list(score_breakdown.values())
    max_values = [15, 65, 20]  # max possible per category, matches breakdown order

    axes[0].bar(labels, values, color="#6C5CE7")
    axes[0].bar(labels, [m - v for m, v in zip(max_values, values)],
                bottom=values, color="#DFE6E9")
    axes[0].set_title(f"Resume Score Breakdown\nTotal: {resume_score}/100")
    axes[0].set_ylabel("Points")
    axes[0].tick_params(axis='x', rotation=20)

    # --- Chart 2: ATS keyword match (pie chart) ---
    axes[1].pie(
        [len(matched_keywords), len(missing_keywords)],
        labels=["Matched", "Missing"],
        autopct="%1.0f%%",
        colors=["#00B894", "#D63031"],
        startangle=90
    )
    axes[1].set_title(f"ATS Keyword Match for '{target_role}'\n({ats_score}% compatible)")

    plt.tight_layout()
    plt.show()

    # --- Text summary report ---
    print("=" * 60)
    print(f"SMART RESUME ANALYZER - REPORT")
    print("=" * 60)
    print(f"File analyzed        : {filename}")
    print(f"Target Role          : {target_role}")
    print(f"Resume Score         : {resume_score}/100")
    print(f"ATS Compatibility    : {ats_score}%")
    print(f"Matched Keywords     : {', '.join(matched_keywords) if matched_keywords else 'None'}")
    print(f"Missing Keywords     : {', '.join(missing_keywords) if missing_keywords else 'None'}")
    print("-" * 60)
    print("Improvement Suggestions:")
    for i, tip in enumerate(feedback_list, start=1):
        print(f"  {i}. {tip}")
    print("=" * 60)


# Render the final dashboard
display_dashboard(
    resume_filename, resume_score, score_breakdown, target_role,
    ats_score, matched_keywords, missing_keywords, feedback_list
)


## Optional: Run the Full Pipeline in One Function
Combines Modules 1-5 into a single callable pipeline (matches the
**Project Workflow** described in the brief: upload -> extract -> process ->
score -> ATS match -> suggestions -> dashboard).


In [ ]:
def run_resume_analyzer_pipeline(role):
    """
    End-to-end pipeline: upload resume -> parse -> score -> ATS check
    -> generate feedback -> display dashboard.

    role: one of the keys in ROLE_KEYWORDS, e.g. "Data Analyst"
    """
    # Step 1: Upload & Parse
    filename, text = upload_and_parse_resume()

    # Step 2: Resume Score Analyzer
    score, breakdown, details = analyze_resume_score(text)

    # Step 3: ATS Keyword Checker
    matched, missing, ats_pct = check_ats_keywords(text, role)

    # Step 4: Smart Feedback System
    feedback = generate_feedback(breakdown, details, missing, ats_pct)

    # Step 5: Dashboard and Report Generation
    display_dashboard(filename, score, breakdown, role, ats_pct, matched, missing, feedback)


# Example usage - uncomment to run interactively with a new upload:
# run_resume_analyzer_pipeline(role="Web Developer")
